# `expansion_hunter_01_run` — ExpansionHunter on Verily Workbench

Run **in a VWB JupyterLab app** (`wb` on `PATH`) after
[`expansion_hunter_00_prep_reference.ipynb`](expansion_hunter_00_prep_reference.ipynb)
(assembly38 FASTA + fai + catalogs). This VM does not share the Locityper
app disk; restage refs here even if the other VM already ran locityper_00.
EH does not need Jellyfish. This is the CLI path around the broken
Workflows GUI.

The copy in this repo is `ExpansionHunterMinicram`
([`expansion_hunter/README.md`](../../expansion_hunter/README.md)): one
`make_minicram_for_expansion_hunter` pass against the Nearline CRAM, then
bw2 ExpansionHunter `--analysis-mode optimized-streaming` on the local
subset. Submission follows Matt’s `eTRs_getPhasedAlleleInfo` notebook:
stage a WDL to a workspace bucket, then `wb workflow job run`.

Docs: [Cromwell in Workbench](https://support.workbench.verily.com/docs/guides/workflows/cromwell/),
[`wb workflow job run`](https://support.workbench.verily.com/docs/references/cli_reference/wb/workflow/job/run/).

## What this notebook does

1. Check `wb` auth / workspace
2. Upload smoke + full catalogs
3. Copy `ExpansionHunterMinicram.wdl` into a workspace GCS bucket
4. `wb workflow create` (`expansion-hunter`) if it is missing
5. Write and upload a **one-sample** inputs JSON (smoke test)
6. Optionally submit that job
7. Optionally build a batch CSV and submit one job per row

`SUBMIT_SINGLE` and `SUBMIT_BATCH` stay **off** until you turn them on.

Cromwell must pull both the print_reads image
(`aou-locityper-print-reads:0.1.0`) and the EH image
(`aou-expansion-hunter:0.1.0`). VPC-SC workspaces need Artifact Registry
mirrors; set `PRINT_READS_DOCKER` / `EH_DOCKER`. After a WDL change, set
`RECREATE_WORKFLOW=True` once.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from typing import Any


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        wdl = p / "expansion_hunter" / "wdl" / "ExpansionHunterMinicram.wdl"
        if wdl.is_file():
            return p
        nested = p / "aou-lr-phase-2" / "expansion_hunter" / "wdl" / "ExpansionHunterMinicram.wdl"
        if nested.is_file():
            return nested.parents[2]
    raise FileNotFoundError(
        "Cannot find expansion_hunter/wdl/ExpansionHunterMinicram.wdl. Clone "
        "kvg/aou-lr-phase-2 into this app (or cd into the clone) and re-run."
    )


def sh(cmd: list[str], *, check: bool = True) -> subprocess.CompletedProcess[str]:
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, check=check, text=True)


def capture(cmd: list[str], *, check: bool = True) -> str:
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, check=check, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout, end="" if proc.stdout.endswith("\n") else "\n")
    if proc.stderr:
        print(proc.stderr, end="" if proc.stderr.endswith("\n") else "\n", file=sys.stderr)
    return proc.stdout


def wb_json(*args: str) -> Any:
    raw = capture(["wb", *args, "--format=JSON"])
    raw = raw.strip()
    if not raw:
        return None
    return json.loads(raw)


def gcs_cp(src: str | Path, dest: str) -> None:
    sh(["gcloud", "storage", "cp", str(src), dest])


REPO_ROOT = find_repo_root()
WDL_LOCAL = REPO_ROOT / "expansion_hunter" / "wdl" / "ExpansionHunterMinicram.wdl"
SMOKE_CATALOG_LOCAL = REPO_ROOT / "expansion_hunter" / "configs" / "smoke.catalog.json"
FULL_CATALOG_LOCAL = REPO_ROOT / "expansion_hunter" / "configs" / "candidate_EH_Loci.GRCh38.json"
COLUMN_MAPPING_LOCAL = REPO_ROOT / "expansion_hunter" / "configs" / "column_mapping.json"
SCRATCH = Path.cwd() / "expansion_hunter_rw_scratch"
SCRATCH.mkdir(parents=True, exist_ok=True)

# Workspace resource ID (not the gs:// name). `wb resource list --type=GCS_BUCKET`.
OUTPUT_BUCKET_ID = os.environ.get("EH_OUTPUT_BUCKET_ID", "aou-lr-phase2-resources")
OUTPUT_BUCKET_GS = os.environ.get("EH_OUTPUT_BUCKET_GS", "gs://aou-lr-phase2-resources/")

WORKFLOW_ID = os.environ.get("EH_WORKFLOW_ID", "expansion-hunter")
WDL_BUCKET_PATH = os.environ.get("EH_WDL_PATH", "wdl/expansion_hunter/ExpansionHunterMinicram.wdl")
OUTPUT_PATH = os.environ.get("EH_OUTPUT_PATH", "workflowRuns/expansion_hunter")
STAGE_PREFIX = os.environ.get("EH_STAGE_PREFIX", "expansion_hunter")

# Smoke-test sample + refs from expansion_hunter_00_prep_reference.
# FASTA defaults to locityper/refs if that VM already uploaded assembly38;
# paste the URIs from the prep notebook if it staged expansion_hunter/refs instead.
SAMPLE_ID = os.environ.get("EH_SAMPLE_ID", "1000000")
CRAM = os.environ.get(
    "EH_CRAM",
    "gs://vwb-aou-datasets-controlled/pooled/wgs/cram/v8_base/wgs_1000000.cram",
)
CRAI = os.environ.get(
    "EH_CRAI",
    "gs://vwb-aou-datasets-controlled/pooled/wgs/cram/v8_base/wgs_1000000.cram.crai",
)
REF_FA = os.environ.get(
    "EH_REF_FA",
    "gs://aou-lr-phase2-resources/locityper/refs/Homo_sapiens_assembly38.fasta",
)
REF_FAI = os.environ.get(
    "EH_REF_FAI",
    "gs://aou-lr-phase2-resources/locityper/refs/Homo_sapiens_assembly38.fasta.fai",
)
CATALOG = os.environ.get(
    "EH_CATALOG",
    "gs://aou-lr-phase2-resources/expansion_hunter/smoke.catalog.json",
)
SEX = os.environ.get("EH_SEX", "female")

WINDOW_SIZE = int(os.environ.get("EH_WINDOW_SIZE", "1000"))
MERGE_REGIONS_DISTANCE = int(os.environ.get("EH_MERGE_REGIONS_DISTANCE", "1000"))
MAX_RETRY = int(os.environ.get("EH_MAX_RETRY", "3"))
WAIT_TIME = int(os.environ.get("EH_WAIT_TIME", "30"))
GCLOUD_PROJECT = os.environ.get("EH_GCLOUD_PROJECT", os.environ.get("GOOGLE_CLOUD_PROJECT", ""))

PRINT_READS_DOCKER = os.environ.get(
    "EH_PRINT_READS_DOCKER",
    "us-central1-docker.pkg.dev/broad-dsp-lrma/aou-lr/aou-locityper-print-reads:0.1.0",
)
EH_DOCKER = os.environ.get(
    "EH_DOCKER",
    "us-central1-docker.pkg.dev/broad-dsp-lrma/aou-lr/aou-expansion-hunter:0.1.0",
)

# Optional path to a local batch CSV (same columns as
# expansion_hunter/configs/batch.header.csv). If empty, the batch cell writes a
# one-row CSV from SAMPLE_ID / CRAM / … in this cell.
BATCH_CSV_LOCAL = os.environ.get("EH_BATCH_CSV", "")

STAGE_CATALOG = True
STAGE_WDL = True
REGISTER_WORKFLOW = True
RECREATE_WORKFLOW = False
SUBMIT_SINGLE = False
SUBMIT_BATCH = False
CANCEL_JOB_ID = ""  # set to a job UUID to cancel in the last cell

print("REPO_ROOT:", REPO_ROOT)
print("WDL_LOCAL:", WDL_LOCAL, "exists=" + str(WDL_LOCAL.is_file()))
print("OUTPUT_BUCKET_ID:", OUTPUT_BUCKET_ID or "(unset)")
print("WORKFLOW_ID:", WORKFLOW_ID)
print("SAMPLE_ID:", SAMPLE_ID or "(unset)")
print("REF_FA:", REF_FA or "(unset)")
print("CATALOG:", CATALOG or "(unset)")
print("SEX:", SEX)
print("PRINT_READS_DOCKER:", PRINT_READS_DOCKER)
print("EH_DOCKER:", EH_DOCKER)
print("GCLOUD_PROJECT:", GCLOUD_PROJECT or "(unset)")
print("SUBMIT_SINGLE:", SUBMIT_SINGLE, "SUBMIT_BATCH:", SUBMIT_BATCH)
print("wb:", shutil.which("wb"))


## Workspace and buckets

`--output-bucket-id` is the **resource ID** from `wb resource list`, not the
`gs://` name. Matt’s eTR notebook used `rw-migration-aou-rw-0b461bba` for a
bucket whose cloud path was `gs://cloned-rw-migration-…`.


In [ ]:
if shutil.which("wb") is None:
    raise SystemExit("wb is not on PATH. Open this notebook in a Verily Workbench Jupyter app.")

sh(["wb", "version"], check=False)
sh(["wb", "auth", "status"], check=False)
sh(["wb", "status"], check=False)

print("\n--- GCS buckets in this workspace ---")
sh(["wb", "resource", "list", "--type=GCS_BUCKET"], check=False)

try:
    wb_json("resource", "list", "--type=GCS_BUCKET")
except Exception as exc:  # noqa: BLE001
    print("could not parse JSON bucket list:", exc)

if not OUTPUT_BUCKET_ID:
    print(
        "\nSet OUTPUT_BUCKET_ID in the config cell to a resource ID from the list above."
    )
else:
    resolved = capture(["wb", "resource", "resolve", f"--id={OUTPUT_BUCKET_ID}"], check=False).strip().splitlines()
    resolved = next((line.strip() for line in reversed(resolved) if line.strip()), "")
    if resolved and not OUTPUT_BUCKET_GS:
        if resolved.startswith("gs://") or resolved.startswith("s3://"):
            OUTPUT_BUCKET_GS = resolved
        elif " " not in resolved and "/" not in resolved.split(":")[0]:
            OUTPUT_BUCKET_GS = f"gs://{resolved}"
        else:
            print("could not parse resolve output; set OUTPUT_BUCKET_GS in the config cell")
    print("OUTPUT_BUCKET_GS:", OUTPUT_BUCKET_GS or "(unset)")


## Stage catalogs

Uploads `smoke.catalog.json` (two chr1 loci) and the 711-locus
`candidate_EH_Loci.GRCh38.json`. Skip if those objects already exist unless
you change the JSON. The smoke URI is the default `CATALOG` for the inputs cell.


In [ ]:
if not OUTPUT_BUCKET_ID or not OUTPUT_BUCKET_GS:
    raise SystemExit("Set OUTPUT_BUCKET_ID (and re-run the bucket cell) before staging.")

SMOKE_CATALOG_URI = f"{OUTPUT_BUCKET_GS.rstrip('/')}/{STAGE_PREFIX}/smoke.catalog.json"
FULL_CATALOG_URI = f"{OUTPUT_BUCKET_GS.rstrip('/')}/{STAGE_PREFIX}/candidate_EH_Loci.GRCh38.json"

if STAGE_CATALOG:
    for src, dest in (
        (SMOKE_CATALOG_LOCAL, SMOKE_CATALOG_URI),
        (FULL_CATALOG_LOCAL, FULL_CATALOG_URI),
    ):
        if not src.is_file():
            raise FileNotFoundError(src)
        gcs_cp(src, dest)
        print("staged catalog:", dest)
else:
    print("STAGE_CATALOG=False; skip copy")
    print("smoke:", SMOKE_CATALOG_URI)
    print("full:", FULL_CATALOG_URI)


## Stage the WDL and register `expansion-hunter`

Workbench cannot register a Git path as a workflow. Copy the WDL into the
workspace bucket, then `wb workflow create`. Re-run with `RECREATE_WORKFLOW=True`
after you change the WDL (that deletes the workspace workflow resource, not
GCS outputs).


In [ ]:
if not OUTPUT_BUCKET_ID or not OUTPUT_BUCKET_GS:
    raise SystemExit("Set OUTPUT_BUCKET_ID (and re-run the bucket cell) before staging.")

WDL_URI = f"{OUTPUT_BUCKET_GS.rstrip('/')}/{WDL_BUCKET_PATH}"
INPUTS_URI = f"{OUTPUT_BUCKET_GS.rstrip('/')}/{STAGE_PREFIX}/inputs.smoke.json"
COLUMN_MAPPING_URI = f"{OUTPUT_BUCKET_GS.rstrip('/')}/{STAGE_PREFIX}/column_mapping.json"
BATCH_CSV_URI_PATH = f"{STAGE_PREFIX}/batch.csv"

if STAGE_WDL:
    gcs_cp(WDL_LOCAL, WDL_URI)
    print("staged WDL:", WDL_URI)
else:
    print("STAGE_WDL=False; skip copy")

print("\n--- workflows already in the workspace ---")
sh(["wb", "workflow", "list"], check=False)

existing_ids: set[str] = set()
try:
    listed = wb_json("workflow", "list")
    rows = listed if isinstance(listed, list) else (listed or {}).get("result") or (listed or {}).get("workflows") or []
    if isinstance(rows, dict):
        rows = [rows]
    for row in rows:
        if not isinstance(row, dict):
            continue
        for key in ("id", "workflowId", "workflow_id", "resourceId"):
            if row.get(key):
                existing_ids.add(str(row[key]))
except Exception as exc:  # noqa: BLE001
    print("could not parse workflow list JSON:", exc)

registered = WORKFLOW_ID in existing_ids
print(f"{WORKFLOW_ID} already registered:", registered)

if RECREATE_WORKFLOW and registered:
    sh(["wb", "workflow", "delete", "--quiet", f"--workflow={WORKFLOW_ID}"])
    registered = False

if REGISTER_WORKFLOW and not registered:
    sh(
        [
            "wb",
            "workflow",
            "create",
            f"--bucket-id={OUTPUT_BUCKET_ID}",
            f"--path={WDL_BUCKET_PATH}",
            f"--workflow={WORKFLOW_ID}",
            "--workflow-type=WDL",
            "--display-name=ExpansionHunter minicram",
            "--description=Per-sample ExpansionHunter (make_minicram + optimized-streaming) for VWB CLI submit",
        ]
    )
elif registered:
    print(f"leave existing workflow {WORKFLOW_ID} in place")

sh(["wb", "workflow", "describe", f"--workflow={WORKFLOW_ID}"], check=False)


## One-sample inputs JSON

`CATALOG` defaults to the smoke catalog URI. Point it at
`…/expansion_hunter/candidate_EH_Loci.GRCh38.json` for the full 711-locus
panel. `sex` is required by ExpansionHunter; smoke default is `female`.


In [ ]:
required = {
    "SAMPLE_ID": SAMPLE_ID,
    "CRAM": CRAM,
    "CRAI": CRAI,
    "REF_FA": REF_FA,
    "REF_FAI": REF_FAI,
    "CATALOG": CATALOG,
    "SEX": SEX,
}
missing = [k for k, v in required.items() if not v]
if missing:
    print("Not uploading yet; set these in the config cell:", ", ".join(missing))
    INPUTS = None
else:
    INPUTS = {
        "ExpansionHunterMinicram.sample_id": SAMPLE_ID,
        "ExpansionHunterMinicram.cram": CRAM,
        "ExpansionHunterMinicram.crai": CRAI,
        "ExpansionHunterMinicram.ref_fa": REF_FA,
        "ExpansionHunterMinicram.ref_fai": REF_FAI,
        "ExpansionHunterMinicram.catalog": CATALOG,
        "ExpansionHunterMinicram.sex": SEX,
        "ExpansionHunterMinicram.window_size": WINDOW_SIZE,
        "ExpansionHunterMinicram.merge_regions_distance": MERGE_REGIONS_DISTANCE,
        "ExpansionHunterMinicram.max_retry": MAX_RETRY,
        "ExpansionHunterMinicram.wait_time": WAIT_TIME,
        "ExpansionHunterMinicram.gcloud_project": GCLOUD_PROJECT,
        "ExpansionHunterMinicram.print_reads_docker": PRINT_READS_DOCKER,
        "ExpansionHunterMinicram.eh_docker": EH_DOCKER,
    }
    inputs_path = SCRATCH / "inputs.smoke.json"
    inputs_path.write_text(json.dumps(INPUTS, indent=2) + "\n")
    print(inputs_path.read_text())
    gcs_cp(inputs_path, INPUTS_URI)
    print("uploaded", INPUTS_URI)


## Submit one job

Turn on `SUBMIT_SINGLE` in the config cell after the inputs upload succeeds.


In [ ]:
if not SUBMIT_SINGLE:
    print("SUBMIT_SINGLE=False; not submitting. Set True in the config cell and re-run.")
elif INPUTS is None:
    raise SystemExit("Fill SAMPLE_ID / CRAM / … before submitting.")
else:
    sh(
        [
            "wb",
            "workflow",
            "job",
            "run",
            f"--workflow={WORKFLOW_ID}",
            f"--output-bucket-id={OUTPUT_BUCKET_ID}",
            f"--output-path={OUTPUT_PATH}",
            f"--inputs-uri={INPUTS_URI}",
            "--delete-intermediate-outputs",
        ]
    )


## Optional: batch CSV

Workbench batch jobs take a CSV in a workspace bucket plus a column map.
Shared files (`catalog`, reference) must be **repeated on every row**.

If `BATCH_CSV_LOCAL` points at a file, that file is uploaded as-is.
Otherwise this cell writes a one-row CSV from the config cell.


In [ ]:
BATCH_FIELDS = [
    "sample_id",
    "cram",
    "crai",
    "ref_fa",
    "ref_fai",
    "catalog",
    "sex",
]

batch_path = SCRATCH / "batch.csv"
if BATCH_CSV_LOCAL:
    src = Path(BATCH_CSV_LOCAL)
    if not src.is_file():
        raise FileNotFoundError(src)
    shutil.copyfile(src, batch_path)
else:
    if missing:
        print("No batch CSV: config sample fields still missing:", ", ".join(missing))
        batch_path = None
    else:
        with batch_path.open("w", newline="") as fh:
            writer = csv.DictWriter(fh, fieldnames=BATCH_FIELDS)
            writer.writeheader()
            writer.writerow(
                {
                    "sample_id": SAMPLE_ID,
                    "cram": CRAM,
                    "crai": CRAI,
                    "ref_fa": REF_FA,
                    "ref_fai": REF_FAI,
                    "catalog": CATALOG,
                    "sex": SEX,
                }
            )

if batch_path is not None:
    print(batch_path.read_text())
    gcs_cp(batch_path, f"{OUTPUT_BUCKET_GS.rstrip('/')}/{BATCH_CSV_URI_PATH}")
    gcs_cp(COLUMN_MAPPING_LOCAL, COLUMN_MAPPING_URI)
    print("batch CSV:", f"{OUTPUT_BUCKET_GS.rstrip('/')}/{BATCH_CSV_URI_PATH}")
    print("column map:", COLUMN_MAPPING_URI)

    if not SUBMIT_BATCH:
        print("SUBMIT_BATCH=False; not submitting the batch.")
    else:
        mapping = ",".join(f"{k}={v}" for k, v in json.loads(COLUMN_MAPPING_LOCAL.read_text()).items())
        sh(
            [
                "wb",
                "workflow",
                "job",
                "run",
                f"--workflow={WORKFLOW_ID}",
                f"--output-bucket-id={OUTPUT_BUCKET_ID}",
                f"--output-path={OUTPUT_PATH}",
                f"--batch-input-bucket-id={OUTPUT_BUCKET_ID}",
                f"--batch-input-csv-path={BATCH_CSV_URI_PATH}",
                f"--column-mapping={mapping}",
                "--delete-intermediate-outputs",
            ]
        )


## Monitor / cancel

Re-run this cell while a job is in flight. Set `CANCEL_JOB_ID` in the config
cell only when you intend to cancel.


In [ ]:
sh(["wb", "workflow", "job", "list", f"--workflow={WORKFLOW_ID}", "--limit=20"], check=False)

job_id = os.environ.get("EH_JOB_ID", "").strip()
if job_id:
    sh(["wb", "workflow", "job", "describe", f"--job-id={job_id}"], check=False)
    sh(["wb", "workflow", "job", "task", "list", f"--job-id={job_id}"], check=False)

if CANCEL_JOB_ID:
    sh(["wb", "workflow", "job", "cancel", f"--job-id={CANCEL_JOB_ID}"])
else:
    print("CANCEL_JOB_ID empty; nothing cancelled.")
